# Validate epoch evaluation and checkpointing on TPU v5e-1

[Open in Colab](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_TPU_Epoch_Eval_Save_V5e1.ipynb)

This notebook checks TrainLM's `accelerator="tpu"` path, which uses the legacy PyTorch/XLA coordinator. It is separate from the native TorchTPU notebooks. In Colab, connect to a user-managed Cloud TPU VM with machine type `ct5lp-hightpu-1t` and a working PyTorch/XLA environment. A standard hosted Colab TPU runtime does not establish v5e-1 hardware. This notebook does not install or replace Python packages.

It runs the focused CPU contract tests, then checks combined epoch evaluation/checkpointing, evaluation-only, checkpoint-only, and resume from the first epoch checkpoint on one TPU device. The model and packed data are created locally. It does not measure performance or certify other TPU generations.


## 1. Get TrainLM source

Run this after the notebook update has merged to `main`. The printed revision is included in the evidence report.


In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

repo_root = Path(tempfile.mkdtemp(prefix="trainlm-xla-epoch-")) / "TrainLM"
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", "main",
     "https://github.com/Dhiraj309/TrainLM.git", str(repo_root)],
    check=True,
)
source_revision = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True
).strip()
sys.path.insert(0, str(repo_root / "src"))
print(f"source_revision={source_revision}")


## 2. Run CPU contract tests

These tests check public argument forwarding, worker flags, and epoch-boundary trainer behavior without initializing a TPU. They are separate from the hardware run below.


In [ ]:
import importlib.util
import os
import subprocess

if importlib.util.find_spec("pytest") is None:
    raise RuntimeError("pytest is unavailable; this notebook does not install packages.")

test_paths = [
    "tests/test_api_tpu_coordinator.py",
    "tests/test_tpu_entrypoint.py",
    "tests/test_api_lifecycle.py",
    "tests/test_finetuning_api.py",
    "tests/contracts/test_roadmap_status.py",
    "tests/training/test_trainer.py",
]
env = os.environ.copy()
env["PYTHONPATH"] = str(repo_root / "src") + os.pathsep + env.get("PYTHONPATH", "")
cpu_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--tb=short", *test_paths],
    cwd=repo_root, env=env, text=True, capture_output=True, timeout=1200,
)
print(cpu_tests.stdout[-6000:])
if cpu_tests.returncode:
    print(cpu_tests.stderr[-3000:])
    raise RuntimeError(f"CPU contract tests failed with exit code {cpu_tests.returncode}.")
print(f"cpu_contract_tests=PASS source_revision={source_revision}")


## 3. Run epoch evaluation, checkpoint, and resume on v5e-1

Before running, confirm the connected VM is `ct5lp-hightpu-1t` in your Cloud TPU details. The coordinator will launch one XLA worker. The model and packed test data are created locally, so no Hub token or model download is needed.


In [ ]:
import importlib.metadata
import importlib.util
import json

import torch
from transformers import GPT2Config, GPT2LMHeadModel
from trainlm import PackedBinDataset, TrainLMTrainer, TrainLMTrainingArguments
from trainlm.data.writer import write_packed_binary_shard

# Confirm this in the Cloud TPU VM page before leaving the value unchanged.
TPU_VM_MACHINE_TYPE = "ct5lp-hightpu-1t"
if TPU_VM_MACHINE_TYPE != "ct5lp-hightpu-1t":
    raise RuntimeError("This validation notebook is scoped to a v5e-1 Cloud TPU VM.")
if importlib.util.find_spec("torch_xla") is None:
    raise RuntimeError(
        "This feature uses PyTorch/XLA. Connect Colab to a v5e-1 Cloud TPU VM "
        "with its supported XLA runtime; this notebook will not install it."
    )

validation_root = Path(tempfile.mkdtemp(prefix="trainlm-epoch-validation-"))
model_dir = validation_root / "tiny-gpt2"
model_dir.mkdir()
model_config = GPT2Config(
    vocab_size=64, n_positions=16, n_ctx=16, n_embd=32, n_layer=1, n_head=2,
    bos_token_id=1, eos_token_id=2, pad_token_id=0, use_cache=False,
)
GPT2LMHeadModel(model_config).save_pretrained(model_dir)

def documents(count):
    return [[3 + ((row + offset) % 60) for offset in range(16)] for row in range(count)]

train_dir = validation_root / "train-data"
eval_dir = validation_root / "eval-data"
write_packed_binary_shard(documents(16), train_dir, shard_id="train", vocab_size=64)
write_packed_binary_shard(documents(4), eval_dir, shard_id="eval", vocab_size=64)
train_data = PackedBinDataset.from_directory(
    train_dir, sequence_length=16, split="train", seed=42
)
eval_data = PackedBinDataset.from_directory(
    eval_dir, sequence_length=16, split="validation"
)
assert len(train_data) == 16 and len(eval_data) == 4

common = dict(
    accelerator="tpu",
    max_steps=6,
    sequence_length=16,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=3,
    learning_rate=1e-3,
    warmup_steps=0,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    max_eval_batches=2,
)
run_dir = validation_root / "fresh"
trainer = TrainLMTrainer.from_pretrained(
    model_dir,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=TrainLMTrainingArguments(output_dir=run_dir, **common),
)
fresh = trainer.train()
fresh_worker = fresh["worker_summary"]
fresh_log = (run_dir / "train.log").read_text(encoding="utf-8")
fresh_eval_count = fresh_log.count('"stage": "evaluation_completed"')
fresh_checkpoints = fresh_worker["committed_checkpoints"]
assert fresh["status"] == "completed"
assert fresh_worker["steps"] == 6 and fresh_worker["micro_steps"] == 16
assert fresh_worker["world_size"] == 1
assert fresh_worker["runtime"]["device"].startswith("xla")
assert fresh_eval_count == 2, f"Expected 2 epoch evaluations, found {fresh_eval_count}."
assert len(fresh_checkpoints) == 2, f"Expected 2 epoch checkpoints, found {len(fresh_checkpoints)}."
train_data.close()
eval_data.close()
first_epoch_checkpoint = run_dir / "checkpoint-3"
assert (first_epoch_checkpoint / "manifest.json").is_file()

# Continue from the first epoch boundary in a separate output directory.
resume_dir = validation_root / "resumed"
resume_train_data = PackedBinDataset.from_directory(
    train_dir, sequence_length=16, split="train", seed=42
)
resume_eval_data = PackedBinDataset.from_directory(
    eval_dir, sequence_length=16, split="validation"
)
resume_trainer = TrainLMTrainer.from_pretrained(
    model_dir,
    train_dataset=resume_train_data,
    eval_dataset=resume_eval_data,
    args=TrainLMTrainingArguments(output_dir=resume_dir, **common),
)
resumed = resume_trainer.train(resume_from_checkpoint=first_epoch_checkpoint)
resume_worker = resumed["worker_summary"]
resume_log = (resume_dir / "train.log").read_text(encoding="utf-8")
resume_eval_count = resume_log.count('"stage": "evaluation_completed"')
assert resumed["status"] == "completed"
assert resume_worker["world_size"] == 1
assert resume_worker["runtime"]["device"].startswith("xla")
assert resume_worker["steps"] == 6 and resume_worker["micro_steps"] == 16
assert resume_eval_count == 1
assert len(resume_worker["committed_checkpoints"]) == 1
resume_train_data.close()
resume_eval_data.close()

def run_independent_schedule(name, eval_strategy, save_strategy):
    train_rows = PackedBinDataset.from_directory(
        train_dir, sequence_length=16, split="train", seed=42
    )
    eval_rows = (
        PackedBinDataset.from_directory(
            eval_dir, sequence_length=16, split="validation"
        )
        if eval_strategy == "epoch"
        else None
    )
    output_dir = validation_root / name
    run_args = {
        **common,
        "max_steps": 3,
        "eval_strategy": eval_strategy,
        "save_strategy": save_strategy,
        "max_eval_batches": 2 if eval_rows is not None else None,
    }
    try:
        schedule_trainer = TrainLMTrainer.from_pretrained(
            model_dir,
            train_dataset=train_rows,
            eval_dataset=eval_rows,
            args=TrainLMTrainingArguments(output_dir=output_dir, **run_args),
        )
        result = schedule_trainer.train()
        worker = result["worker_summary"]
        log = (output_dir / "train.log").read_text(encoding="utf-8")
        evaluations = log.count('"stage": "evaluation_completed"')
        checkpoints = worker["committed_checkpoints"]
        expected_evaluations = int(eval_strategy == "epoch")
        expected_checkpoints = int(save_strategy == "epoch")
        assert result["status"] == "completed"
        assert worker["world_size"] == 1
        assert worker["steps"] == 3 and worker["micro_steps"] == 8
        assert evaluations == expected_evaluations
        assert len(checkpoints) == expected_checkpoints
        return {
            "status": result["status"],
            "steps": worker["steps"],
            "micro_steps": worker["micro_steps"],
            "epoch_evaluations": evaluations,
            "committed_checkpoints": [Path(path).name for path in checkpoints],
        }
    finally:
        train_rows.close()
        if eval_rows is not None:
            eval_rows.close()

evaluation_only = run_independent_schedule(
    "evaluation-only", eval_strategy="epoch", save_strategy="no"
)
checkpoint_only = run_independent_schedule(
    "checkpoint-only", eval_strategy="no", save_strategy="epoch"
)

report = {
    "schema_version": 1,
    "scope": "legacy_xla_epoch_eval_save_v5e1",
    "source_revision": source_revision,
    "hardware": {
        "vm_machine_type_entered_by_owner": TPU_VM_MACHINE_TYPE,
        "hardware_identity_verified_by_notebook": False,
    },
    "runtime": {
        "torch": torch.__version__,
        "torch_xla": importlib.metadata.version("torch-xla"),
        "training_backend": "pytorch_xla",
    },
    "cpu_contract_tests": {
        "status": "passed",
        "exit_code": cpu_tests.returncode,
        "summary": cpu_tests.stdout.strip().splitlines()[-1],
    },
    "fresh_run": {
        "xla_device": fresh_worker["runtime"]["device"],
        "status": fresh["status"],
        "steps": fresh_worker["steps"],
        "micro_steps": fresh_worker["micro_steps"],
        "epoch_evaluations": fresh_eval_count,
        "committed_checkpoints": [Path(path).name for path in fresh_checkpoints],
    },
    "evaluation_only": evaluation_only,
    "checkpoint_only": checkpoint_only,
    "resume_run": {
        "status": resumed["status"],
        "steps": resume_worker["steps"],
        "micro_steps": resume_worker["micro_steps"],
        "epoch_evaluations": resume_eval_count,
        "committed_checkpoints": [
            Path(path).name for path in resume_worker["committed_checkpoints"]
        ],
        "resumed_from": first_epoch_checkpoint.name,
    },
    "performance_certified": False,
}
report_path = validation_root / "tpu_epoch_validation_report.json"
report_path.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
print(f"evidence_report={report_path}")


## 4. Download the report

The report records source revision, package versions, evaluation counts, and checkpoint names. The VM type is owner-entered metadata; verify it in Cloud TPU details before sharing the report. Training logs and temporary files remain in the runtime.


In [ ]:
try:
    from google.colab import files
except ImportError:
    print(f"Download this report from: {report_path}")
else:
    files.download(str(report_path))
